# Notebook 01 — Exploratory Data Analysis
**Project:** Medical Insurance Cost Prediction  
**Dataset:** [Kaggle — Medical Cost Personal Dataset](https://www.kaggle.com/datasets/mirichoi0218/insurance)  
**Goal:** Understand the data structure, distributions, and key relationships before modelling.

---
### Flow
1. Load & inspect data
2. Univariate analysis (distributions)
3. Bivariate analysis (relationships with target)
4. Correlation analysis
5. Key findings summary

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

from src.data_loader import load_raw_data
df = load_raw_data()
df.head()

## 1. Data Overview

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nData Types:\n{df.dtypes}')
print(f'\nMissing Values:\n{df.isnull().sum()}')
print(f'\nDuplicates: {df.duplicated().sum()}')

In [ ]:
df.describe().round(2)

## 2. Univariate Analysis — Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
num_cols = ['age', 'bmi', 'charges']
colors = ['steelblue', 'seagreen', 'tomato']

for ax, col, color in zip(axes, num_cols, colors):
    ax.hist(df[col], bins=30, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2, label=f'Mean: {df[col].mean():.1f}')
    ax.set_title(f'Distribution of {col.title()}')
    ax.set_xlabel(col)
    ax.legend()

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
cat_cols = ['sex', 'smoker', 'region']

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    ax.bar(counts.index, counts.values, color='steelblue', edgecolor='white')
    ax.set_title(f'{col.title()} Distribution')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, str(v), ha='center', fontsize=10)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/02_categorical_distributions.png', bbox_inches='tight')
plt.show()

## 3. Bivariate Analysis — Relationships with Charges

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Age vs Charges
axes[0].scatter(df['age'], df['charges'], alpha=0.4, color='steelblue', s=20)
axes[0].set_title('Age vs Charges')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Charges ($)')

# BMI vs Charges
axes[1].scatter(df['bmi'], df['charges'], alpha=0.4, color='seagreen', s=20)
axes[1].set_title('BMI vs Charges')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Charges ($)')

# Children vs Charges
axes[2].scatter(df['children'], df['charges'], alpha=0.4, color='tomato', s=20)
axes[2].set_title('Children vs Charges')
axes[2].set_xlabel('Number of Children')
axes[2].set_ylabel('Charges ($)')

plt.suptitle('Numerical Features vs Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_numerical_vs_charges.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['sex', 'smoker', 'region']):
    groups = [df[df[col] == val]['charges'] for val in df[col].unique()]
    ax.boxplot(groups, labels=df[col].unique(), patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(f'{col.title()} vs Charges')
    ax.set_xlabel(col)
    ax.set_ylabel('Charges ($)')

plt.suptitle('Categorical Features vs Insurance Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_categorical_vs_charges.png', bbox_inches='tight')
plt.show()

In [ ]:
# Smoker impact — the most important finding
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE plot: smoker vs non-smoker charges
for label, grp in df.groupby('smoker'):
    axes[0].hist(grp['charges'], bins=30, alpha=0.6, label=f'Smoker: {label}', density=True)
axes[0].set_title('Charge Distribution: Smokers vs Non-Smokers')
axes[0].set_xlabel('Charges ($)')
axes[0].legend()

# BMI vs Charges coloured by smoker
colors_map = {'yes': 'tomato', 'no': 'steelblue'}
for label, grp in df.groupby('smoker'):
    axes[1].scatter(grp['bmi'], grp['charges'], alpha=0.5, s=20,
                    color=colors_map[label], label=f'Smoker: {label}')
axes[1].set_title('BMI vs Charges (by Smoking Status)')
axes[1].set_xlabel('BMI')
axes[1].set_ylabel('Charges ($)')
axes[1].legend()

plt.suptitle('Key Insight: Smoking Status Drives Charges', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_smoker_impact.png', bbox_inches='tight')
plt.show()

print('Mean charges by smoking status:')
print(df.groupby('smoker')['charges'].mean().round(2))

## 4. Correlation Analysis

In [ ]:
df_encoded = df.copy()
df_encoded['sex']    = df_encoded['sex'].map({'male': 1, 'female': 0})
df_encoded['smoker'] = df_encoded['smoker'].map({'yes': 1, 'no': 0})
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True)

corr = df_encoded.corr()

plt.figure(figsize=(10, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print('\nTop correlations with charges:')
print(corr['charges'].sort_values(ascending=False).round(3))

## 5. Key Findings

| Finding | Detail |
|---|---|
| **Smoking is the #1 driver** | Smokers pay 3-4x more on average (~$32,000 vs ~$8,400) |
| **BMI × Smoking interaction** | Smokers with BMI > 30 form the highest-cost cluster |
| **Age has strong positive correlation** | Older individuals pay progressively more |
| **Region has minimal impact** | Charges are relatively uniform across regions |
| **Gender has negligible effect** | Male vs female charges are nearly identical |
| **Charges are right-skewed** | Most people pay moderate amounts; a small group pays very high |
| **No missing values** | Dataset is clean; 1 duplicate removed |

> **Next step:** `02_Feature_Engineering.ipynb` — encode categoricals, create interaction features, and prepare final feature matrix.